# DR-OPIC Tutorial

This notebook explains the DR-OPIC framework end to end: the idea, the math, the Python API, the CLI, and the artifact contract. It uses only tiny deterministic examples. No private datasets or model weights are required.

**DR-OPIC** means **Domain-Routed On-Policy Iterative Correction**. The framework is for coding SLM experiments where the student model attempts first, verifiers expose failures, repairs are verified, and training data is built from reachable failure states.

## 1. Setup

From the repository root, install the package in editable mode:

```bash
python -m pip install -e ".[dev]"
```

Then this notebook can import `dr_opic` directly.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from dr_opic.maths import smoothed_pass_rate, zpd_weight, group_relative_advantages, selector_gap
from dr_opic.verifier import PythonTask, verify_python, static_check_python
from dr_opic.schemas import Task, Candidate
from dr_opic.forge import rollout_python_task, repair_failures, build_round_artifacts, save_round_artifacts
from dr_opic.delta import token_delta_spans
from dr_opic.routing import route_task
from dr_opic.safety import classify_coding_safety, selective_accept
from dr_opic.compression import estimate_dense_model
from dr_opic.datasets import audit_rows

## 2. The Core Loop

The DR-OPIC loop is:

1. Route a task to a bounded coding domain or abstain.
2. Let the current student attempt the task before any teacher answer is used.
3. Run executable verification.
4. Compute task learnability with ZPD weighting.
5. Repair failed attempts using the failed code and verifier observation.
6. Verify the repair.
7. Select the most learnable verified winner.
8. Emit JSON/JSONL artifacts for training and release review.

## 3. Math: ZPD Weighting

For `s` passes in `K` samples, DR-OPIC uses a Jeffreys-smoothed pass estimate:

$$\tilde{p} = \frac{s + 0.5}{K + 1}$$

The zone-of-proximal-development weight is:

$$w_{zpd} = 4\tilde{p}(1 - \tilde{p})$$

This peaks near tasks the model sometimes solves and sometimes fails. It downweights already-mastered tasks and tasks that need decomposition before direct training.

In [ ]:
for passes, samples in [(0, 4), (1, 4), (2, 4), (4, 4)]:
    print({
        "passes": passes,
        "samples": samples,
        "p_tilde": round(smoothed_pass_rate(passes, samples), 4),
        "zpd_weight": round(zpd_weight(passes, samples), 4),
    })

## 4. Math: Verifier Reward

The framework uses a scalar reward for ranking and group-relative training:

$$r(y, x) = 1.00\,final\_pass + 0.25\,public\_fraction + 0.10\,syntax\_ok + 0.05\,import\_ok - penalties$$

Final pass dominates. Partial reward exists to rank failures and near misses; it is not a substitute for held-out tests.

## 5. Verify Python Code

`verify_python` extracts code, runs static guards, writes a temporary candidate file, executes tests in a subprocess, and returns a structured result. This is not a security sandbox; use a container or VM for untrusted model output.

In [ ]:
py_task = PythonTask(
    prompt="Implement reverse_words(s) returning words in reverse order.",
    entrypoint="reverse_words",
    tests="assert reverse_words('one two three') == 'three two one'\nassert reverse_words('solo') == 'solo'",
)

bad_code = "def reverse_words(s):\n    return s\n"
good_code = "def reverse_words(s):\n    return ' '.join(reversed(s.split()))\n"

print("bad:", verify_python(bad_code, py_task))
print("good:", verify_python(good_code, py_task))

In [ ]:
unsafe = "import os\ndef run():\n    os.system('echo no')\n"
static_check_python(unsafe, entrypoint="run")

## 6. Run A Student-First Forge Round

A generator callback stands in for the current student model. A repair callback stands in for a teacher, repair model, or deterministic postprocessor. In a real run these callbacks would call a model or service.

In [ ]:
task = Task(
    task_id="reverse_words_demo",
    prompt="Implement reverse_words(s) returning words in reverse order.",
    entrypoint="reverse_words",
    tests="assert reverse_words('one two three') == 'three two one'\nassert reverse_words('solo') == 'solo'",
)

def student(task: Task, index: int) -> str:
    if index == 0:
        return "def reverse_words(s):\n    return s\n"
    return "def reverse_words(s):\n    return ' '.join(reversed(s.split()))\n"

def repair(task: Task, failed: Candidate, round_index: int) -> str:
    return "def reverse_words(s):\n    return ' '.join(reversed(s.split()))\n"

group = rollout_python_task(task, student, k=2)
repairs = repair_failures(group, repair, rounds=1)
artifacts = build_round_artifacts(group, repairs)

artifacts.keys(), artifacts["passes"], artifacts["zpd_weight"], artifacts["winner"]["source"]

In [ ]:
artifacts["winner"]

## 7. Math: Group-Relative Advantages

For one rollout group with rewards $r_i$:

$$A_i = \frac{r_i - mean(r)}{std(r) + \epsilon}$$

A low-compute self-training objective can behavior-clone only positive-advantage samples:

$$L_{self} = -w_{task}\sum_i max(A_i, 0)\log \pi_\theta(y_i|x)$$

In [ ]:
rewards = [row["reward"] for row in artifacts["rollouts"]]
group_relative_advantages(rewards)

## 8. Math: Learnable Winner Selection

Given a verified candidate `c` and failed attempt `s`, DR-OPIC scores:

$$Score(c;s) = \lambda_v pass(c) + \lambda_f fuzz(c) + \lambda_l \frac{\log p(c|x)}{|c|} - \lambda_e edit(c,s) - \lambda_c complexity(c) - \lambda_d deps(c)$$

The point is not to choose the most elegant teacher answer. The point is to choose the verified answer closest to the student's reachable failure state.

## 9. Math: Delta-Span Training

Whole-answer negative training can punish correct shared code. Delta-span training compares failed code `y-` and fixed code `y+`:

- `D+`: inserted or replaced spans in the fix
- `D-`: removed or replaced spans in the failure

$$L_\Delta = -\sum_{t \in D+}\log p_\theta(y^+_t) + \lambda_{neg}\sum_{t \in D-} max(0, \log p_\theta(y^-_t) - \log p_{ref}(y^-_t) - m)$$

The implementation produces the spans used to build this training record.

In [ ]:
token_delta_spans(bad_code, good_code)

## 10. Save Artifacts

A professional run should save stable artifacts. The repo's artifact contract writes:

- `round_summary.json`
- `student_rollouts.jsonl`
- `verified_repairs.jsonl`
- `learnable_winner.json`
- `delta_spans.json`

In [ ]:
with TemporaryDirectory() as tmp:
    paths = save_round_artifacts(tmp, artifacts)
    for name, path in paths.items():
        print(name, Path(path).name, Path(path).stat().st_size)

## 11. Routing, Safety, And Abstention

Domain routing and safety checks decide whether the SLM should answer at all. A specialist should abstain outside its validated domain.

In [ ]:
prompt = "Fix this Python traceback and add pytest coverage."
route = route_task(prompt)
safety = classify_coding_safety(prompt)
print(route)
print(safety)
print("accepted:", selective_accept(route.accepted, safety.allowed, route.confidence))

## 12. Dataset Audits

The repo does not include training data. The audit helper validates external rows before training. SFT rows must be exactly `prompt` and `response`; preference rows must include `prompt`, `chosen`, and `rejected`.

In [ ]:
rows = [
    {"prompt": "Implement inc(x) for integers.", "response": "def inc(x):\n    return x + 1\n"},
]
audit_rows(rows, schema="sft")

## 13. Cost And Compression Estimates

For a dense transformer with `N` parameters, rough compute and memory estimates are:

$$C_{train} \approx 6ND$$

$$C_{infer/token} \approx 2N$$

$$M_{weights} \approx bN$$

Compression is a release phase after behavior improves, not the first source of capability.

In [ ]:
estimate_dense_model(3.09e9)

## 14. CLI Cheatsheet

Run from the repository root:

```bash
python -m dr_opic.cli forge-demo
python -m dr_opic.cli --output outputs\demo forge-demo
python -m dr_opic.cli verify-python examples\python_task.json --code examples\reverse_words_good.py
python -m dr_opic.cli verify-python examples\python_task.json --code examples\reverse_words_bad.py --fail-on-error
python -m dr_opic.cli zpd --passes 2 --samples 5
python -m dr_opic.cli route "Fix this Python traceback and add pytest coverage"
python -m dr_opic.cli estimate-model --params 3.09e9
python -m dr_opic.cli audit-jsonl C:\datasets\slm\sft.jsonl --schema sft
```

## 15. Release Checklist

Do not promote a run just because one number moved. A proper DR-OPIC release should report:

- greedy@1
- coverage@K
- selected@K
- selector gap
- repair@1
- hard-subset pass rate
- malformed output rate
- repeated-token collapse rate
- OOD false accept rate
- unsafe compliance rate
- latency and memory
- contamination and dataset audit summary

The core falsifiable claim is simple: student-first repair and delta-span training should improve selected@K or repair@1 without damaging formatting, safety, or abstention.